# At-risk feature EDA

This notebook reads from MySQL through the read-only `ml_readonly` account. `v_ml_features` contains only values available at the middle of a term. The outcome is intentionally kept in `v_ml_labels`: final grade F and detention are known only at term end and must not be used as input features. Keeping the label in a separate view makes accidental label leakage visible during training and review.

The current schema has no assignment-submission table or fee due-date column. The feature view documents the safe proxies used: an assignment is submitted when a marks row exists, and pending fee delay is measured from term start to the mid-term snapshot.


In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

load_dotenv()
user = os.getenv('ML_DB_USER', 'ml_readonly')
password = os.getenv('ML_DB_PASSWORD', '')
host = os.getenv('ML_DB_HOST', 'localhost')
port = os.getenv('ML_DB_PORT', '3306')
database = os.getenv('ML_DB_NAME', 'edu_db')
engine = create_engine(f'mysql+pymysql://{user}:{password}@{host}:{port}/{database}')


In [ ]:
features = pd.read_sql(text('SELECT * FROM v_ml_features'), engine)
labels = pd.read_sql(text('SELECT * FROM v_ml_labels'), engine)
df = features.merge(labels, on=['student_id', 'term_id'], how='left')
df['at_risk'] = df['at_risk'].fillna(0).astype(int)
df.head()


## Class balance

In [ ]:
class_balance = df['at_risk'].value_counts(dropna=False).rename_axis('at_risk').reset_index(name='rows')
class_balance['rate_pct'] = class_balance['rows'] / len(df) * 100
class_balance


## Feature distributions

In [ ]:
numeric = df.select_dtypes(include=np.number).drop(columns=['student_id', 'term_id', 'at_risk'], errors='ignore')
numeric.hist(figsize=(16, 12), bins=20)
plt.tight_layout()


## Correlation heatmap

In [ ]:
plt.figure(figsize=(14, 10))
sns.heatmap(df.select_dtypes(include=np.number).corr(), cmap='coolwarm', center=0)
plt.title('Feature and label correlations')
plt.show()


## At-risk rate by attendance band

In [ ]:
bands = [-np.inf, 60, 75, 85, 100, np.inf]
labels_band = ['<60', '60-75', '75-85', '85-100', '100+']
df['attendance_band'] = pd.cut(df['attendance_pct_midterm'], bins=bands, labels=labels_band, right=False)
attendance_risk = df.groupby('attendance_band', observed=False)['at_risk'].agg(['count', 'mean']).rename(columns={'mean': 'at_risk_rate'})
attendance_risk['at_risk_rate_pct'] = attendance_risk['at_risk_rate'] * 100
attendance_risk


## IQR outlier report

In [ ]:
outlier_rows = []
for column in numeric.columns:
    series = numeric[column].dropna()
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    mask = (numeric[column] < lower) | (numeric[column] > upper)
    outlier_rows.append({'feature': column, 'q1': q1, 'q3': q3, 'lower': lower, 'upper': upper, 'outlier_count': int(mask.sum()), 'outlier_rate_pct': float(mask.mean() * 100)})
iqr_report = pd.DataFrame(outlier_rows).sort_values('outlier_count', ascending=False)
iqr_report
